# CNN Assignment: Vegetable Image Classification Using Deep Learning

**Dataset:** [Kaggle Vegetable Image Dataset](https://www.kaggle.com/datasets/misrakahmed/vegetable-image-dataset)  
**Classes (15):** Bean, Bitter Gourd, Bottle Gourd, Brinjal, Broccoli, Cabbage, Capsicum, Carrot, Cauliflower, Cucumber, Papaya, Potato, Pumpkin, Radish, Tomato

## Part 1: Import Required Libraries

In [ ]:
# Data handling
import os
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully!")

TensorFlow version: 2.20.0
All libraries imported successfully!


## Part 2: Define Dataset Paths

In [ ]:
# Define dataset paths (Kaggle environment)
BASE_DIR = "/kaggle/input/vegetable-image-dataset/Vegetable Images"

TRAIN_DIR      = os.path.join(BASE_DIR, "train")
TEST_DIR       = os.path.join(BASE_DIR, "test")
VALIDATION_DIR = os.path.join(BASE_DIR, "validation")

print("Train path      :", TRAIN_DIR)
print("Test path       :", TEST_DIR)
print("Validation path :", VALIDATION_DIR)

# Quick sanity check
for path in [TRAIN_DIR, TEST_DIR, VALIDATION_DIR]:
    exists = os.path.exists(path)
    print(f"  Exists ({os.path.basename(path)}): {exists}")

Train path      : /kaggle/input/vegetable-image-dataset/Vegetable Images/train
Test path       : /kaggle/input/vegetable-image-dataset/Vegetable Images/test
Validation path : /kaggle/input/vegetable-image-dataset/Vegetable Images/validation
  Exists (train): False
  Exists (test): False
  Exists (validation): False


## Part 3: Count Images in Each Class

In [ ]:
def count_images_per_class(dataset_path):
    """
    Counts the number of images in each class folder.

    Args:
        dataset_path (str): Path to the dataset split folder.

    Returns:
        dict: {class_name: image_count}
    """
    counts = {}
    class_folders = sorted(os.listdir(dataset_path))

    for class_name in class_folders:
        class_path = os.path.join(dataset_path, class_name)
        if os.path.isdir(class_path):
            images = [
                f for f in os.listdir(class_path)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ]
            counts[class_name] = len(images)

    return counts


# Apply to each split
train_counts      = count_images_per_class(TRAIN_DIR)
test_counts       = count_images_per_class(TEST_DIR)
validation_counts = count_images_per_class(VALIDATION_DIR)

print("Train counts:")
for cls, cnt in train_counts.items():
    print(f"  {cls}: {cnt}")

print("\nTest counts:")
for cls, cnt in test_counts.items():
    print(f"  {cls}: {cnt}")

print("\nValidation counts:")
for cls, cnt in validation_counts.items():
    print(f"  {cls}: {cnt}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/vegetable-image-dataset/Vegetable Images/train'

## Part 4: Create a Summary DataFrame

In [ ]:
# Build individual DataFrames
train_df      = pd.DataFrame(list(train_counts.items()),      columns=["Vegetable", "Train Count"])
test_df       = pd.DataFrame(list(test_counts.items()),       columns=["Vegetable", "Test Count"])
validation_df = pd.DataFrame(list(validation_counts.items()), columns=["Vegetable", "Validation Count"])

# Merge into one summary DataFrame
summary_df = (
    train_df
    .merge(test_df,       on="Vegetable")
    .merge(validation_df, on="Vegetable")
)

print("Dataset Summary:")
print(summary_df.to_string(index=False))

print(f"\nTotal train images     : {summary_df['Train Count'].sum()}")
print(f"Total test images      : {summary_df['Test Count'].sum()}")
print(f"Total validation images: {summary_df['Validation Count'].sum()}")

## Part 5: Visualize Class Distribution

In [ ]:
x = np.arange(len(summary_df["Vegetable"]))
width = 0.28

fig, ax = plt.subplots(figsize=(16, 6))

bars1 = ax.bar(x - width, summary_df["Train Count"],      width, label="Train",      color="steelblue")
bars2 = ax.bar(x,         summary_df["Test Count"],       width, label="Test",       color="tomato")
bars3 = ax.bar(x + width, summary_df["Validation Count"], width, label="Validation", color="seagreen")

ax.set_title("Number of Images per Vegetable Class (Train / Test / Validation)", fontsize=15, fontweight="bold")
ax.set_xlabel("Vegetable Class", fontsize=12)
ax.set_ylabel("Number of Images", fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(summary_df["Vegetable"], rotation=45, ha="right", fontsize=10)
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

print("""
Is the dataset balanced?
------------------------
Yes, the dataset is balanced. Each vegetable class contains the same number of images
across the train, test, and validation splits. A balanced dataset ensures the model
does not become biased toward any particular class during training.
""")

## Part 6: Image Preprocessing and Augmentation

In [ ]:
# Training generator — with augmentation + rescaling
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2   # 80/20 split from training folder
)

# Validation / test generator — rescaling ONLY (no augmentation)
val_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2
)

print("Image data generators configured.")

## Part 7: Load Images Using `flow_from_directory`

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42

# Training dataset
train_ds = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    seed=SEED
)

# Validation dataset
val_ds = val_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    seed=SEED
)

print("\nClass indices:")
for cls, idx in train_ds.class_indices.items():
    print(f"  {idx}: {cls}")

## Part 8: Build the CNN Model

In [ ]:
model = Sequential([

    # ── Convolution Block 1 ──────────────────────────────
    Conv2D(32, (3, 3), activation="relu", input_shape=(224, 224, 3)),
    MaxPooling2D(pool_size=(2, 2)),

    # ── Convolution Block 2 ──────────────────────────────
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),

    # ── Convolution Block 3 ──────────────────────────────
    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),

    # ── Fully Connected Layers ───────────────────────────
    Flatten(),
    Dense(512, activation="relu"),
    Dropout(0.5),

    # ── Output Layer (15 classes) ────────────────────────
    Dense(15, activation="softmax")
], name="Vegetable_CNN")

model.summary()

## Part 9: Compile the Model

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("""
Model Architecture Explanation
================================
1. Convolutional Layers (3 blocks):
   The model uses 3 Conv2D layers with 32, 64, and 128 filters respectively.
   Each layer learns increasingly complex spatial features — from edges and
   textures (block 1) to high-level shapes and patterns (block 3).

2. Pooling Layers:
   MaxPooling2D reduces the spatial dimensions (height × width) by half after
   each conv block. This lowers computational cost, reduces overfitting, and
   makes the model more robust to small positional shifts in features.

3. Dropout (rate=0.5):
   Randomly deactivates 50% of neurons during each training step. This forces
   the network to learn redundant representations and is a powerful regularizer
   that helps prevent overfitting.

4. Softmax Output:
   Softmax converts raw scores (logits) into a probability distribution over
   the 15 classes. Each output value is in [0, 1] and all values sum to 1,
   making it ideal for multi-class classification.
""")

## Part 10: Train the Model

In [ ]:
# EarlyStopping callback
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

print("""
Why EarlyStopping is useful:
EarlyStopping monitors the validation loss after every epoch. If it does not
improve for 'patience' consecutive epochs (here: 5), training halts automatically
and the model weights are restored to the best checkpoint. This prevents
overfitting and saves unnecessary compute time.
""")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stopping]
)

## Part 11: Evaluate the Model

In [ ]:
# Evaluate on training data
train_loss, train_acc = model.evaluate(train_ds, verbose=0)

# Evaluate on validation data
val_loss, val_acc = model.evaluate(val_ds, verbose=0)

print(f"Training   Accuracy : {train_acc:.4f}")
print(f"Validation Accuracy : {val_acc:.4f}")
print(f"Training   Loss     : {train_loss:.4f}")
print(f"Validation Loss     : {val_loss:.4f}")

gap = train_acc - val_acc
print(f"\nAccuracy gap (train - val): {gap:.4f}")

print("""
Overfitting Analysis:
---------------------
If the training accuracy is significantly higher than the validation accuracy
(gap > ~5–10%), the model is overfitting — it has memorised the training data
rather than learning generalisable patterns.
A small gap indicates good generalisation. Techniques like Dropout, data
augmentation, and EarlyStopping all help reduce overfitting.
""")

## Part 12: Plot Accuracy and Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history.history["accuracy"]) + 1)

# ── Accuracy Curve ────────────────────────────────────────
axes[0].plot(epochs_range, history.history["accuracy"],     label="Training Accuracy",   color="steelblue",  linewidth=2)
axes[0].plot(epochs_range, history.history["val_accuracy"], label="Validation Accuracy", color="tomato",     linewidth=2, linestyle="--")
axes[0].set_title("Training vs Validation Accuracy", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(linestyle="--", alpha=0.5)

# ── Loss Curve ────────────────────────────────────────────
axes[1].plot(epochs_range, history.history["loss"],     label="Training Loss",   color="steelblue",  linewidth=2)
axes[1].plot(epochs_range, history.history["val_loss"], label="Validation Loss", color="tomato",     linewidth=2, linestyle="--")
axes[1].set_title("Training vs Validation Loss", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(linestyle="--", alpha=0.5)

plt.suptitle("Model Training History", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()